# Helper notebook for week 2 of the course extra readings

Related to article [The Hitchhiker’s Guide to Feature Extraction](https://www.kdnuggets.com/2019/06/hitchhikers-guide-feature-extraction.html)

## Using Featuretools to create features from relational data

In [11]:
from pydoc import describe

import featuretools as ft
from woodwork.logical_types import PostalCode, Categorical
import warnings
import pandas as pd

data = ft.demo.load_mock_customer()

customers_df = data["customers"]
sessions_df = data["sessions"]
transactions_df = data["transactions"]

from woodwork.logical_types import Categorical, PostalCode

dataframes = {
    "customers": (
        customers_df,
        "customer_id",
        None,  # No time index
        {"zip_code": PostalCode},
    ),
    "sessions": (
        sessions_df,
        "session_id",
        "session_start",
        {"device": Categorical},
    ),
    "transactions": (
        transactions_df,
        "transaction_id",
        "transaction_time",
        {"product_id": Categorical},
    ),
}

relationships = [
    ("sessions", "session_id", "transactions", "session_id"),
    ("customers", "customer_id", "sessions", "customer_id"),
]

with warnings.catch_warnings():
    # Suppress the FutureWarning about SeriesGroupBy in featuretools
    warnings.filterwarnings(
        "ignore",
        message=r"The provided callable .* is currently using SeriesGroupBy\.",
        category=FutureWarning,
    )
    feature_matrix_customers, features_defs = ft.dfs(
        dataframes=dataframes,
        relationships=relationships,
        target_dataframe_name="customers",
    )

feature_matrix_customers

,zip_code,COUNT(sessions),MODE(sessions.device),NUM_UNIQUE(sessions.device),COUNT(transactions),MAX(transactions.amount),MEAN(transactions.amount),MIN(transactions.amount),MODE(transactions.product_id),NUM_UNIQUE(transactions.product_id),...,STD(sessions.SKEW(transactions.amount)),STD(sessions.SUM(transactions.amount)),SUM(sessions.MAX(transactions.amount)),SUM(sessions.MEAN(transactions.amount)),SUM(sessions.MIN(transactions.amount)),SUM(sessions.NUM_UNIQUE(transactions.product_id)),SUM(sessions.SKEW(transactions.amount)),SUM(sessions.STD(transactions.amount)),MODE(transactions.sessions.device),NUM_UNIQUE(transactions.sessions.device)
customer_id,,,,,,,,,,,,,,,,,,,,,
1,60091,8,mobile,3,126,139.43,71.631905,5.81,4,5,...,0.589386,279.510713,1057.97,582.193117,78.59,40.0,-0.476122,312.745952,mobile,3
2,13244,7,desktop,3,93,146.81,77.422366,8.73,4,5,...,0.509798,251.609234,931.63,548.905851,154.60,35.0,-0.277640,258.700528,desktop,3
3,13244,6,desktop,3,93,149.15,67.060430,5.89,1,5,...,0.429374,219.021420,847.63,405.237462,66.21,29.0,2.286086,257.299895,desktop,3
4,60091,8,mobile,3,109,149.95,80.070459,5.73,2,5,...,0.387884,235.992478,1157.99,649.657515,131.51,37.0,0.002764,356.125829,mobile,3
5,60091,6,mobile,3,79,149.02,80.375443,7.55,5,5,...,0.415426,402.775486,839.76,472.231119,86.49,30.0,0.014384,259.873954,mobile,3


In [12]:
examples = [
    "SUM(sessions.STD(transactions.amount))",
    "STD(sessions.SUM(transactions.amount))",
]

display(feature_matrix_customers[examples].sort_index())

,SUM(sessions.STD(transactions.amount)),STD(sessions.SUM(transactions.amount))
customer_id,,
1,312.745952,279.510713
2,258.700528,251.609234
3,257.299895,219.021420
4,356.125829,235.992478
5,259.873954,402.775486


## Using Pandas to create features from relational data

In [15]:
session_stats = (
    transactions_df.groupby("session_id")["amount"]
    .agg(
        total_amount="sum",
        amount_std="std", # This doesn't work: amount_std=lambda values: values.std(ddof=0),
    )
    .reset_index()
)

# Match sessions by their actual IDs
session_stats = sessions_df[["session_id", "customer_id"]].merge(
    session_stats,
    on="session_id",
    how="left",
    validate="one_to_one",
)

session_stats["total_amount"] = session_stats["total_amount"].fillna(0)

display(session_stats.sort_values("customer_id").head(20))

,session_id,customer_id,total_amount,amount_std
17,18,1,943.28,36.044179
3,4,1,1613.93,40.187205
28,29,1,1420.09,32.324534
5,6,1,1266.60,42.654755
26,27,1,894.09,44.354104
25,26,1,809.97,30.450261
8,9,1,1052.03,46.905665
13,14,1,1025.63,39.825249
32,33,2,804.83,28.298415
30,31,2,1240.19,39.414339


Example for customer 1:

In [16]:
customer_1_df = session_stats[session_stats["customer_id"] == 1]

display(customer_1_df)

,session_id,customer_id,total_amount,amount_std
3,4,1,1613.93,40.187205
5,6,1,1266.60,42.654755
8,9,1,1052.03,46.905665
13,14,1,1025.63,39.825249
17,18,1,943.28,36.044179
25,26,1,809.97,30.450261
26,27,1,894.09,44.354104
28,29,1,1420.09,32.324534


In [18]:
customer_1_df.describe()

,session_id,customer_id,total_amount,amount_std
count,8.000000,8.0,8.000000,8.000000
mean,16.625000,1.0,1128.202500,39.093244
std,9.913015,0.0,279.510713,5.768646
min,4.000000,1.0,809.970000,30.450261
25%,8.250000,1.0,930.982500,35.114267
50%,16.000000,1.0,1038.830000,40.006227
75%,26.250000,1.0,1304.972500,43.079592
max,29.000000,1.0,1613.930000,46.905665


In [19]:
total_amount_std_sum = customer_1_df["amount_std"].sum()
print(f"Total amount std sum for customer 1: {total_amount_std_sum}")

Total amount std sum for customer 1: 312.74595150692755


we can see that:

- the intersection of `total_amount` and `std` is 279.510713, which matches Feature Tools.
- the sum of `amount_std` is 312.74, which also matches Feature Tools.

In [20]:
std_total_amount = customer_1_df["total_amount"].std()
print(f"Standard deviation of total amounts for customer 1: {std_total_amount}")

Standard deviation of total amounts for customer 1: 279.510712840952
